In [9]:
import pandas as pd
import numpy as np
import pickle
import ast
import re
import emoji
from catboost import CatBoostRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 1. INITIALISATION & FONCTIONS (SANS BUG)
# ---------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()
hashtag_regex = re.compile(r'#\w+')
mention_regex = re.compile(r'@\w+')
emoji_set = set(emoji.EMOJI_DATA)
ski_slang_map = {"sick": "amazing", "gnarly": "impressive", "insane": "incredible", 
                 "crushing": "performing well", "send it": "go for it", "fail": "funny mistake"}

def get_sentiment(text):
    text = str(text).lower()
    for word, replacement in ski_slang_map.items():
        text = text.replace(word, replacement)
    return analyzer.polarity_scores(text)['compound']

def engineer_features(df, is_train=True, train_df=None):
    df = df.copy()
    if 'id' in df.columns: df = df.rename(columns={'id': 'ID'})
    df['ID'] = df['ID'].astype(str).str.strip()
        
    df['description'] = df['description'].fillna('')
    df['desc_len'] = df['description'].str.len()
    df['word_count'] = df['description'].str.split().str.len()
    df['hashtag_count'] = df['description'].apply(lambda x: len(hashtag_regex.findall(str(x))))
    df['mention_count'] = df['description'].apply(lambda x: len(mention_regex.findall(str(x))))
    df['emoji_count'] = df['description'].apply(lambda x: sum(1 for char in str(x) if char in emoji_set))
    df['sent_compound'] = df['description'].apply(get_sentiment)

    df['release_year'] = df['release_year'].fillna(df['release_year'].median() if is_train else train_df['release_year'].median())
    df['video_age'] = 2026 - df['release_year']
    
    # Extraction temporelle rapide et sécurisée
    df['post_month'] = 6
    numeric_ids = pd.to_numeric(df['ID'], errors='coerce')
    valid_mask = numeric_ids.notna()
    if valid_mask.any():
        valid_ints = numeric_ids[valid_mask].values.astype(np.int64)
        shifted = np.right_shift(valid_ints, 32)
        dates = pd.to_datetime(shifted, unit='s', errors='coerce')
        df.loc[valid_mask, 'post_month'] = dates.month
    df['post_month'] = df['post_month'].fillna(6).astype(int)

    df['artist_count'] = df['artists'].apply(lambda x: len(ast.literal_eval(x)) if pd.notna(x) and str(x).startswith('[') else 1)
    df['is_optimal_duration'] = df['video_duration'].between(6, 12).astype(int) if 'video_duration' in df.columns else 0
    df['is_minimalist'] = (df['desc_len'] <= 20).astype(int)
    df['is_val_thorens'] = df['channel'].str.contains('Val Thorens', na=False, case=False).astype(int)

    if is_train and 'popularity' in df.columns:
        alpha = 5.0
        global_mean = np.log1p(df['popularity']).mean()
        channel_stats = df.groupby('channel')['popularity'].agg(sum_log=lambda x: np.log1p(x).sum(), count='count')
        channel_stats['channel_authority'] = (channel_stats['sum_log'] + alpha * global_mean) / (channel_stats['count'] + alpha)
        df = df.merge(channel_stats[['channel_authority']], on='channel', how='left')
        df['channel_authority'] = df['channel_authority'].fillna(global_mean)
    elif not is_train and train_df is not None:
        channel_authority_map = train_df.groupby('channel')['channel_authority'].first()
        df['channel_authority'] = df['channel'].map(channel_authority_map).fillna(train_df['channel_authority'].mean())
    else:
        df['channel_authority'] = 0

    return df

# ---------------------------------------------------------
# 2. CHARGEMENT & FUSION
# ---------------------------------------------------------
print("Chargement des données...")
X_train = pd.read_csv('X_train.csv', sep=';', skipinitialspace=True).rename(columns=lambda x: x.strip())
y_train = pd.read_csv('y_train.csv', sep=';', skipinitialspace=True).rename(columns=lambda x: x.strip())
X_test = pd.read_csv('X_test.csv', sep=';', skipinitialspace=True).rename(columns=lambda x: x.strip())

for df in [X_train, y_train, X_test]:
    if 'id' in df.columns: df.rename(columns={'id': 'ID'}, inplace=True)
    df['ID'] = df['ID'].astype(str).str.strip()

train_data = engineer_features(X_train.merge(y_train, on='ID', how='inner'), is_train=True)
test_data = engineer_features(X_test, is_train=False, train_df=train_data)

# TF-IDF
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
tfidf_cols = [f'tfidf_{col}' for col in tfidf.fit(train_data['description']).get_feature_names_out()]
train_data = pd.concat([train_data, pd.DataFrame(tfidf.transform(train_data['description']).toarray(), columns=tfidf_cols, index=train_data.index)], axis=1)
test_data = pd.concat([test_data, pd.DataFrame(tfidf.transform(test_data['description']).toarray(), columns=tfidf_cols, index=test_data.index)], axis=1)

print("Chargement multimodal...")
df_audio = pd.read_csv('audio_features_ready.csv').rename(columns=lambda x: x.strip())
df_visual = pd.read_csv('features_visual2.csv').rename(columns=lambda x: x.strip())
df_objects = pd.read_csv('object_detection_features.csv').rename(columns=lambda x: x.strip())
with open('embeddings_video.pkl', 'rb') as f:
    df_emb = pd.DataFrame.from_dict(pickle.load(f), orient='index')
    df_emb.columns = [f'emb_{i}' for i in range(df_emb.shape[1])]
    df_emb.index.name = 'ID'
    df_emb.reset_index(inplace=True)

for df in [df_audio, df_visual, df_emb]:
    id_col = [c for c in df.columns if c.lower() == 'id']
    if id_col: df.rename(columns={id_col[0]: 'ID'}, inplace=True)
    df['ID'] = df['ID'].astype(str).str.strip()

if 'video_id' in df_objects.columns:
    df_objects['ID'] = df_objects['video_id'].astype(str).str.replace('VIDEO_', '', regex=False)
    df_objects.drop(columns=['video_id'], inplace=True)
else:
    df_objects.rename(columns={df_objects.columns[0]: 'ID'}, inplace=True)
    df_objects['ID'] = df_objects['ID'].astype(str).str.replace('VIDEO_', '', regex=False)
df_objects['ID'] = df_objects['ID'].astype(str).str.strip()

def merge_multimodal(df_base):
    return df_base.merge(df_audio, on='ID', how='left').merge(df_visual, on='ID', how='left').merge(df_objects, on='ID', how='left').merge(df_emb, on='ID', how='left')

train_full = merge_multimodal(train_data)
test_full = merge_multimodal(test_data)

# ---------------------------------------------------------
# 3. PRÉPARATION CATBOOST
# ---------------------------------------------------------
print("Préparation pour CatBoost...")
y_train_log = np.log1p(train_full['popularity'])
cols_to_drop = ['ID', 'popularity', 'artists', 'release_year', 'filepath', 'download_timing', 'uploader', 'uploader_short', 'vid', 'uid']
features = [c for c in train_full.columns if c not in cols_to_drop]

X_train_final = train_full[features].copy()
X_test_final = test_full[features].copy()

text_features = ['description'] if 'description' in features else []
cat_features = [c for c in X_train_final.select_dtypes(include=['object', 'string']).columns if c not in text_features]

# Remplissage robuste pour CatBoost
for col in cat_features:
    X_train_final[col] = X_train_final[col].fillna('Unknown').astype(str).str.strip()
    X_test_final[col] = X_test_final[col].fillna('Unknown').astype(str).str.strip()
for col in text_features:
    X_train_final[col] = X_train_final[col].fillna('').astype(str)
    X_test_final[col] = X_test_final[col].fillna('').astype(str)

# ---------------------------------------------------------
# 4. ENTRAÎNEMENT & VALIDATION CATBOOST
# ---------------------------------------------------------
model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    cat_features=cat_features,
    text_features=text_features,
    random_seed=42,
    verbose=0 # On coupe les logs pour la validation croisée
)

print("--- ÉVALUATION LOCALE (5-FOLD CV) ---")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_final)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_log.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[val_idx], y_train_log.iloc[val_idx]
    
    model.fit(X_tr, y_tr)
    rmse = np.sqrt(mean_squared_error(y_va, model.predict(X_va)))
    rmse_scores.append(rmse)
    print(f"Fold {fold+1} - RMSE: {rmse:.4f}")

print(f"✅ RMSE Moyen Local : {np.mean(rmse_scores):.4f}")

print("Génération de la soumission finale...")
model.fit(X_train_final, y_train_log) # Entraînement final sur 100% des données
preds = np.clip(np.expm1(model.predict(X_test_final)), 0, None)

pd.DataFrame({'ID': test_full['ID'], 'popularity': preds}).to_csv('submission_catboost_optim.csv', index=False)
print("✅ Fichier 'submission_catboost_optim.csv' prêt !")

Chargement des données...
Chargement multimodal...
Préparation pour CatBoost...
--- ÉVALUATION LOCALE (5-FOLD CV) ---
Fold 1 - RMSE: 0.1300
Fold 2 - RMSE: 0.1307
Fold 3 - RMSE: 0.1275
Fold 4 - RMSE: 0.1405
Fold 5 - RMSE: 0.1371
✅ RMSE Moyen Local : 0.1331
Génération de la soumission finale...
✅ Fichier 'submission_catboost_optim.csv' prêt !
